# Mars Rover Terrain Segmentation

This notebook runs the repository end to end in a Kaggle-friendly way.

- The first code cell can either use a repo already present in Kaggle or clone it from GitHub.
- It then adds the repo folder to `sys.path` so the local packages import correctly.
- It uses the AI4MARS MSL folder directly: `AI4MARSv0-6/ai4mars-dataset-merged-0.6/msl`.
- It loads both `mcam` and `ncam` for training, and uses the labeled `ncam` test folder for validation and test.
- It runs the project dataset adapter, model factory, Lightning module, training, checkpoint loading, and evaluation.

For a full run on Kaggle, attach the dataset folder or set `AI4MARS_ROOT` to its path, then increase the batch limits and epochs in the config cell.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("MARS_SEG_REPO_URL", "https://github.com/Dalphan/Mars-Rover-Scene-Understanding-and-Health-Monitoring.git")
REPO_DIR = Path(os.environ.get("MARS_SEG_REPO_DIR", "/kaggle/working/mars-rover-terrain-segmentation"))


def is_repo_root(path: Path) -> bool:
    return (path / "configs").exists() and (path / "train").exists()


def find_repo_root() -> Path | None:
    candidates = [Path.cwd(), *Path.cwd().parents, REPO_DIR, Path("/kaggle/working")]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(path for path in input_root.iterdir() if path.is_dir())
    for path in candidates:
        if is_repo_root(path):
            return path
    return None


repo_root = find_repo_root()
if repo_root is None:
    if not REPO_DIR.exists():
        print(f"Cloning repository into {REPO_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    repo_root = REPO_DIR

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Added to sys.path: {str(repo_root) in sys.path}")


In [ ]:
!pip install -r "/kaggle/working/mars-rover-terrain-segmentation/requirements.txt" # ignore

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pytorch_lightning as pl
import torch
from omegaconf import OmegaConf
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

work_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else repo_root
output_dir = work_root / "outputs" / "kaggle_mars_segmentation"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {repo_root}")
print(f"Output dir: {output_dir}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
DATASET_MODE = "ai4mars"
MAX_EPOCHS = 10
RUN_AUDIT = False
RUN_SANITY_CHECK = True


AI4MARS_ROOT = Path(os.environ.get("AI4MARS_ROOT", "/kaggle/input"))


def find_ai4mars_root() -> Path:
    path = AI4MARS_ROOT / "datasets" / "ai4marsv0-6" / "ai4mars-dataset-merged-0.6" / "msl"
    if (path / "mcam" / "images").exists() and (path / "ncam" / "images" / "edr").exists():
        return path
    raise FileNotFoundError(
        "AI4MARS MSL folder not found: "
        f"{path}. Set AI4MARS_ROOT to the parent folder that contains AI4MARSv0-6."
    )


def build_cfg(data_root: Path) -> OmegaConf:
    data_cfg = {
        "name": "ai4mars",
        "taxonomy": "core",
        "root": str(data_root),
        "use_rover_mask_as_label": False,
        "num_classes": 3,
        "ignore_index": 255,
        "loader": {"batch_size": 2, "num_workers": "auto"},
        "image_size": [128, 128],
        "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        "augmentations": {
            "enabled": True,
            "rotation": 10,
            "hflip": 0.5,
            "vflip": 0.0,
            "brightness": 0.2,
            "contrast": 0.2,
            "noise": 0.02,
            "blur": 0.1,
        },
        "train": {"name": "train"},
        "val": {"name": "val"},
        "test": {"name": "test"},
    }

    cfg = {
        "seed": 42,
        "num_classes": 3,
        "ignore_index": 255,
        "paths": {"root": str(repo_root), "output_dir": str(output_dir)},
        "logging": {"save_visuals": True, "visuals_every_n_epochs": 1, "visuals_max_samples": 2},
        "optimizer": {"lr": 6e-5, "weight_decay": 0.01},
        "ckpt_path": None,
        "data": data_cfg,
        "model": {
            "name": "segformer_b0",
            "pretrained": False,
            "checkpoint": None,
            "loss": {"type": "ce", "dice_weight": 0.0},
        },
        "trainer": {
            "max_epochs": MAX_EPOCHS,
            "accelerator": "auto",
            "devices": "auto",
            "precision": "32-true",
            "log_every_n_steps": 1,
            "check_val_every_n_epoch": 1,
            "enable_model_summary": False,
        },
    }
    return OmegaConf.create(cfg)


data_root = find_ai4mars_root()
print(f"Using AI4MARS MSL root: {data_root}")


In [ ]:
import sys
import importlib

# Remove cached local modules
for module in list(sys.modules):
    if (
        module.startswith("mars_datasets")
        or module.startswith("models")
        or module.startswith("train")
        or module.startswith("utils")
    ):
        del sys.modules[module]

# Reimport everything
from mars_datasets.ai4mars import audit_ai4mars
from mars_datasets.datamodule import SegmentationDataModule
from models.factory import build_model
from train.segmentation_module import SegmentationModule
from utils.runtime import configure_runtime
from utils.seed import set_seed
from utils.visualization import plot_image_and_mask

In [ ]:
from mars_datasets.ai4mars import audit_ai4mars
from mars_datasets.datamodule import SegmentationDataModule
from models.factory import build_model
from train.segmentation_module import SegmentationModule
from utils.epoch_logging import EpochMetricsPrinter
from utils.runtime import configure_runtime
from utils.seed import set_seed
from utils.visualization import plot_image_and_mask

cfg = build_cfg(data_root)
runtime_info = configure_runtime(cfg)
set_seed(cfg.seed)

if RUN_AUDIT:
    reports = [audit_ai4mars(root=cfg.data.root, split=split, taxonomy=cfg.data.taxonomy) for split in ("train", "val", "test")]
    print(json.dumps(reports, indent=2))

data_module = SegmentationDataModule(cfg)

if RUN_SANITY_CHECK:
    data_module.setup("fit")

    def plot_training_sample(batch, index=0):
        plot_image_and_mask(
            batch["image"][index],
            batch["mask"][index],
            mean=cfg.data.normalize.mean,
            std=cfg.data.normalize.std,
            ignore_index=cfg.ignore_index,
            num_classes=cfg.num_classes,
        )

    train_batch = next(iter(data_module.train_dataloader()))
    print({key: tuple(value.shape) if hasattr(value, "shape") else type(value) for key, value in train_batch.items() if key in {"image", "mask"}})
    print(train_batch["mask"][0, :8, :8])
    plot_training_sample(train_batch)

model = build_model(cfg)
module = SegmentationModule(cfg, model)
if RUN_SANITY_CHECK:
    module.eval()
    with torch.no_grad():
        logits = module(train_batch["image"][:1])
    print(f"Logits shape: {tuple(logits.shape)}")
    module.train()


In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.paths.output_dir,
    filename="{epoch:03d}-{val_miou:.4f}",
    monitor="val_miou",
    mode="max",
    save_last=True,
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")
logger = TensorBoardLogger(save_dir=cfg.paths.output_dir, name="lightning_logs")

trainer = pl.Trainer(
    **OmegaConf.to_container(cfg.trainer, resolve=True),
    callbacks=[checkpoint_callback, lr_monitor, EpochMetricsPrinter()],
    logger=logger,
    default_root_dir=cfg.paths.output_dir,
    deterministic=False,
    enable_progress_bar=True,
)

trainer.fit(module, datamodule=data_module)
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"Last checkpoint: {checkpoint_callback.last_model_path}")

In [ ]:
best_ckpt = checkpoint_callback.best_model_path or checkpoint_callback.last_model_path
if not best_ckpt:
    raise RuntimeError("No checkpoint was saved during training.")

eval_model = build_model(cfg)
eval_module = SegmentationModule(cfg, eval_model)
state = torch.load(best_ckpt, map_location="cpu")
eval_module.load_state_dict(state["state_dict"], strict=True)

eval_trainer = pl.Trainer(
    **OmegaConf.to_container(cfg.trainer, resolve=True),
    callbacks=[EpochMetricsPrinter()],
    logger=False,
    enable_checkpointing=False,
    deterministic=False,
)
test_results = eval_trainer.test(eval_module, datamodule=data_module, verbose=False)
print(json.dumps(test_results, indent=2))

artifacts = sorted(path.relative_to(cfg.paths.output_dir).as_posix() for path in Path(cfg.paths.output_dir).rglob("*") if path.is_file())
print("Saved artifacts:")
for artifact in artifacts:
    print("-", artifact)